# Conjunto de datos completo sin clusterización

In [1]:
#Importaciones
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

#Lectura de datos
datos = pd.read_excel('03_Clusterizacion.xlsx')
datos.head(24)

,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,Cluster GMM
0,2022-09-01 00:00:00,0.000000,19,7,77,0,4,15,0,Noche,Noche
1,2022-09-01 01:00:00,0.000000,19,7,82,0,4,16,1,Noche,Noche
2,2022-09-01 02:00:00,0.000000,18,9,85,0,3,16,2,Noche,Noche
3,2022-09-01 03:00:00,0.000000,18,11,87,0,3,16,3,Noche,Noche
4,2022-09-01 04:00:00,0.000000,18,11,88,0,3,16,4,Noche,Noche
5,2022-09-01 05:00:00,0.000000,17,15,86,0,3,14,5,Noche,Noche
6,2022-09-01 06:00:00,0.000000,18,47,89,0,3,16,6,Nublado,Lluvioso
7,2022-09-01 07:00:00,6.584959,18,51,95,0,4,17,7,Nublado,Lluvioso
8,2022-09-01 08:00:00,560.422022,18,47,100,0,3,18,8,Nublado,Lluvioso
9,2022-09-01 09:00:00,7720.582326,18,5,100,1,4,18,9,Nublado,Lluvioso


In [2]:
datos["Generacion_prev_hour"] = datos["Generación"].shift(1)
datos["Generacion_prev_day"] = datos["Generación"].shift(24)
datos = datos.dropna(how="any", axis= 0)

Definimos X y y

In [3]:
datos_dia = datos[datos["Cluster GMM"] == "Soleado"].copy()
datos_dia.head(10)

,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day
37,2022-09-02 13:00:00,20596.278869,23,0,45,5,2,11,13,Soleado,Soleado,27523.885172,25478.471342
207,2022-09-09 15:00:00,26098.851182,24,0,43,5,2,11,15,Soleado,Soleado,24394.058046,18959.491592
208,2022-09-09 16:00:00,28500.000000,25,0,39,4,2,10,16,Soleado,Soleado,26098.851182,16494.208425
209,2022-09-09 17:00:00,28500.000000,25,0,38,3,2,10,17,Soleado,Soleado,28500.000000,8718.476156
210,2022-09-09 18:00:00,28500.000000,26,0,36,2,2,10,18,Soleado,Soleado,28500.000000,23968.849012
211,2022-09-09 19:00:00,17303.091873,25,0,37,1,2,9,19,Soleado,Soleado,28500.000000,17559.996819
212,2022-09-09 20:00:00,2230.131403,24,0,40,0,2,9,20,Soleado,Soleado,17303.091873,2087.543919
229,2022-09-10 13:00:00,28500.000000,22,0,53,10,2,12,13,Soleado,Soleado,28976.805233,14247.046020
230,2022-09-10 14:00:00,28500.000000,23,0,47,11,2,11,14,Soleado,Soleado,28500.000000,24394.058046
231,2022-09-10 15:00:00,27067.944242,24,0,43,10,2,11,15,Soleado,Soleado,28500.000000,26098.851182


In [4]:
columns = datos_dia.drop(columns=["Fecha", "Generación", "Cluster KMeans", "Cluster GMM"]).columns

In [5]:
X = datos_dia[columns]
X

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
37,23,0,45,5,2,11,13,27523.885172,25478.471342
207,24,0,43,5,2,11,15,24394.058046,18959.491592
208,25,0,39,4,2,10,16,26098.851182,16494.208425
209,25,0,38,3,2,10,17,28500.000000,8718.476156
210,26,0,36,2,2,10,18,28500.000000,23968.849012
...,...,...,...,...,...,...,...,...,...
18280,25,0,34,5,3,8,15,25399.000000,25443.000000
18281,26,0,31,4,2,8,16,25562.000000,25385.000000
18282,26,0,32,2,1,8,17,25386.000000,22664.000000
18283,25,0,33,1,1,8,18,22872.000000,15736.000000


In [6]:
y = datos_dia[['Generación']]
y

,Generación
37,20596.278869
207,26098.851182
208,28500.000000
209,28500.000000
210,28500.000000
...,...
18280,25562.000000
18281,25386.000000
18282,22872.000000
18283,15825.000000


Dividimos entrenamiento, validación y prueba

In [7]:
train_size = int(0.7 * len(X))
val_size = int(0.85 * len(X))

In [8]:
# Entrenamiento, validación y prueba, 75, 15 y 15
X_train, y_train =  X.iloc[:train_size, :], y.iloc[:train_size, :]
X_val, y_val = X.iloc[train_size:val_size, :], y.iloc[train_size:val_size, :]
X_test, y_test = X.iloc[val_size:, :],  y.iloc[val_size:,:]

print(f'X_train: {len(X_train)}, y_train: {len(y_train)}')
print(f'X_val: {len(X_val)}, y_val: {len(y_val)}')
print(f'X_test: {len(X_test)}, y_test: {len(y_test)}')

X_train: 2412, y_train: 2412
X_val: 517, y_val: 517
X_test: 518, y_test: 518


## Escalar con MinMaxScaler

In [9]:
from sklearn.preprocessing import MinMaxScaler

In [10]:
x_scaler = MinMaxScaler().fit(X_train)
x_scaler

MinMaxScaler()

In [11]:
X_train_scaled = x_scaler.transform(X_train)
print(X_train_scaled)
print(X_train_scaled.shape)

[[0.39130435 0.         0.73076923 ... 0.46666667 0.91746284 0.84928238]
 [0.43478261 0.         0.69230769 ... 0.6        0.81313527 0.63198305]
 [0.47826087 0.         0.61538462 ... 0.66666667 0.86996171 0.54980695]
 ...
 [0.30434783 0.         0.40384615 ... 0.46666667 0.76026667 0.73196667]
 [0.34782609 0.         0.40384615 ... 0.53333333 0.73196667 0.70156667]
 [0.39130435 0.         0.42307692 ... 0.6        0.70156667 0.7206    ]]
(2412, 9)


In [12]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
X_train_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
37,0.391304,0.0,0.730769,0.357143,0.333333,0.6875,0.466667,0.917463,0.849282
207,0.434783,0.0,0.692308,0.357143,0.333333,0.6875,0.600000,0.813135,0.631983
208,0.478261,0.0,0.615385,0.285714,0.333333,0.6250,0.666667,0.869962,0.549807
209,0.478261,0.0,0.596154,0.214286,0.333333,0.6250,0.733333,0.950000,0.290616
210,0.521739,0.0,0.557692,0.142857,0.333333,0.6250,0.800000,0.950000,0.798962
...,...,...,...,...,...,...,...,...,...
12286,0.260870,0.0,0.423077,0.000000,0.000000,0.1250,1.000000,0.000000,0.000000
12301,0.217391,0.0,0.442308,0.357143,0.000000,0.0625,0.400000,0.783667,0.760267
12302,0.304348,0.0,0.403846,0.357143,0.000000,0.1250,0.466667,0.760267,0.731967
12303,0.347826,0.0,0.403846,0.357143,0.000000,0.1875,0.533333,0.731967,0.701567


In [13]:
X_val_scaled = x_scaler.transform(X_val)
print(X_val_scaled)
print(X_val_scaled.shape)

[[0.47826087 0.         0.44230769 ... 0.66666667 0.7034     0.7193    ]
 [0.39130435 0.         0.57692308 ... 0.73333333 0.65726667 0.65726667]
 [0.34782609 0.         0.71153846 ... 0.8        0.6976     0.31023333]
 ...
 [0.82608696 0.         0.09615385 ... 0.46666667 0.77843333 0.77773333]
 [0.34782609 0.         0.34615385 ... 0.2        0.48006667 0.69496667]
 [0.56521739 0.         0.21153846 ... 0.26666667 0.81946667 0.7502    ]]
(517, 9)


In [14]:
X_val_scaled_df = pd.DataFrame(X_val_scaled, index=X_val.index, columns=X_val.columns)
X_val_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
12305,0.478261,0.0,0.442308,0.214286,0.000000,0.3750,0.666667,0.703400,0.719300
12306,0.391304,0.0,0.576923,0.142857,0.000000,0.5000,0.733333,0.657267,0.657267
12307,0.347826,0.0,0.711538,0.071429,0.000000,0.5625,0.800000,0.697600,0.310233
12423,0.217391,0.0,0.788462,0.357143,0.333333,0.5000,0.533333,0.792533,0.490433
12424,0.304348,0.0,0.634615,0.285714,0.333333,0.4375,0.600000,0.783967,0.504200
...,...,...,...,...,...,...,...,...,...
14940,0.608696,0.0,0.250000,0.642857,0.000000,0.2500,0.333333,0.750200,0.785633
14941,0.739130,0.0,0.153846,0.857143,0.000000,0.1250,0.400000,0.766967,0.786967
14942,0.826087,0.0,0.096154,1.000000,0.000000,0.0000,0.466667,0.778433,0.777733
14962,0.347826,0.0,0.346154,0.214286,0.000000,0.0625,0.200000,0.480067,0.694967


In [15]:
X_test_scaled = x_scaler.transform(X_test)
print(X_test_scaled)
print(X_test_scaled.shape)

[[0.69565217 0.         0.11538462 ... 0.33333333 0.92343333 0.76696667]
 [0.82608696 0.         0.03846154 ... 0.4        0.95496667 0.77843333]
 [0.91304348 0.         0.         ... 0.46666667 0.95866667 0.78513333]
 ...
 [0.52173913 0.         0.48076923 ... 0.73333333 0.8462     0.75546667]
 [0.47826087 0.         0.5        ... 0.8        0.7624     0.52453333]
 [0.39130435 0.         0.59615385 ... 0.86666667 0.5275     0.0469    ]]
(518, 9)


In [16]:
X_test_scaled_df = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)
X_test_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
14964,0.695652,0.0,0.115385,0.642857,0.000000,0.0625,0.333333,0.923433,0.766967
14965,0.826087,0.0,0.038462,0.857143,0.000000,0.2500,0.400000,0.954967,0.778433
14966,0.913043,0.0,0.000000,1.000000,0.000000,0.3750,0.466667,0.958667,0.785133
14967,0.956522,0.0,-0.019231,0.857143,0.000000,0.3750,0.533333,0.960967,0.775267
14968,1.000000,0.0,-0.019231,0.642857,0.000000,0.3750,0.600000,0.860433,0.873567
...,...,...,...,...,...,...,...,...,...
18280,0.478261,0.0,0.519231,0.357143,0.666667,0.5000,0.600000,0.846633,0.848100
18281,0.521739,0.0,0.461538,0.285714,0.333333,0.5000,0.666667,0.852067,0.846167
18282,0.521739,0.0,0.480769,0.142857,0.000000,0.5000,0.733333,0.846200,0.755467
18283,0.478261,0.0,0.500000,0.071429,0.000000,0.5000,0.800000,0.762400,0.524533


In [17]:
x_scaller_all = MinMaxScaler().fit(X)
print(x_scaller_all)

MinMaxScaler()


In [18]:
X_scaled = x_scaller_all.transform(X)
print(X_scaled)
print(X_scaled.shape)

[[0.36       0.         0.73584906 ... 0.46666667 0.91746284 0.84928238]
 [0.4        0.         0.69811321 ... 0.6        0.81313527 0.63198305]
 [0.44       0.         0.62264151 ... 0.66666667 0.86996171 0.54980695]
 ...
 [0.48       0.         0.49056604 ... 0.73333333 0.8462     0.75546667]
 [0.44       0.         0.50943396 ... 0.8        0.7624     0.52453333]
 [0.36       0.         0.60377358 ... 0.86666667 0.5275     0.0469    ]]
(3447, 9)


In [19]:
X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)
X_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
37,0.36,0.0,0.735849,0.357143,0.333333,0.6875,0.466667,0.917463,0.849282
207,0.40,0.0,0.698113,0.357143,0.333333,0.6875,0.600000,0.813135,0.631983
208,0.44,0.0,0.622642,0.285714,0.333333,0.6250,0.666667,0.869962,0.549807
209,0.44,0.0,0.603774,0.214286,0.333333,0.6250,0.733333,0.950000,0.290616
210,0.48,0.0,0.566038,0.142857,0.333333,0.6250,0.800000,0.950000,0.798962
...,...,...,...,...,...,...,...,...,...
18280,0.44,0.0,0.528302,0.357143,0.666667,0.5000,0.600000,0.846633,0.848100
18281,0.48,0.0,0.471698,0.285714,0.333333,0.5000,0.666667,0.852067,0.846167
18282,0.48,0.0,0.490566,0.142857,0.000000,0.5000,0.733333,0.846200,0.755467
18283,0.44,0.0,0.509434,0.071429,0.000000,0.5000,0.800000,0.762400,0.524533


In [20]:
y_scaler = MinMaxScaler().fit(y_train)
print(y_scaler)

MinMaxScaler()


In [21]:
y_train_scaled = y_scaler.transform(y_train)
print(y_train_scaled)
print(y_train_scaled.shape)

[[0.68654263]
 [0.86996171]
 [0.95      ]
 ...
 [0.73196667]
 [0.70156667]
 [0.7034    ]]
(2412, 1)


In [22]:
y_train_scaled_df = pd.DataFrame(y_train_scaled, index=y_train.index, columns=y_train.columns)
y_train_scaled_df

,Generación
37,0.686543
207,0.869962
208,0.950000
209,0.950000
210,0.950000
...,...
12286,0.000000
12301,0.760267
12302,0.731967
12303,0.701567


In [23]:
y_val_scaled = y_scaler.transform(y_val)
print(y_val_scaled)
print(y_val_scaled.shape)

[[6.57266667e-01]
 [6.97600000e-01]
 [2.98400000e-01]
 [7.83966667e-01]
 [7.82933333e-01]
 [7.82600000e-01]
 [7.53100000e-01]
 [4.56700000e-01]
 [4.34666667e-02]
 [0.00000000e+00]
 [1.00000000e+00]
 [1.00000000e+00]
 [1.00000000e+00]
 [1.00000000e+00]
 [1.00000000e+00]
 [1.00000000e+00]
 [1.00000000e+00]
 [2.00766667e-01]
 [0.00000000e+00]
 [0.00000000e+00]
 [1.00000000e+00]
 [2.29400000e-01]
 [1.00000000e+00]
 [1.00000000e+00]
 [1.00000000e+00]
 [1.00000000e+00]
 [1.00000000e+00]
 [6.56666667e-01]
 [7.13833333e-01]
 [6.27166667e-01]
 [6.26333333e-01]
 [6.26066667e-01]
 [5.27166667e-01]
 [3.19666667e-01]
 [2.64000000e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [6.57766667e-01]
 [6.34033333e-01]
 [6.36166667e-01]
 [6.55400000e-01]
 [7.37566667e-01]
 [6.63633333e-01]
 [6.37766667e-01]
 [8.02466667e-01]
 [7.83966667e-01]
 [8.05566667e-01]
 [8.13566667e-01]
 [7.97433333e-01]
 [4.59833333e-01]
 [4.69333333e-02]
 [0.00000000e+00]
 [8.12700000e-01]
 [7.91266667e-01]
 [7.82000000e-01]
 [7.752000

In [24]:
y_val_scaled_df = pd.DataFrame(y_val_scaled, index=y_val.index, columns=y_val.columns)
y_val_scaled_df

,Generación
12305,0.657267
12306,0.697600
12307,0.298400
12423,0.783967
12424,0.782933
...,...
14940,0.766967
14941,0.778433
14942,0.785133
14962,0.819467


In [25]:
y_test_scaled = y_scaler.transform(y_test)
print(y_test_scaled)
print(y_test_scaled.shape)

[[0.95496667]
 [0.95866667]
 [0.96096667]
 [0.86043333]
 [0.85716667]
 [0.74293333]
 [0.73216667]
 [0.59493333]
 [0.21896667]
 [0.00806667]
 [0.        ]
 [0.93476667]
 [0.942     ]
 [0.92893333]
 [0.9326    ]
 [0.9653    ]
 [0.95383333]
 [0.85996667]
 [0.72946667]
 [0.2737    ]
 [0.9442    ]
 [0.93683333]
 [0.92893333]
 [0.9326    ]
 [0.97063333]
 [0.95473333]
 [0.8543    ]
 [0.71996667]
 [0.2737    ]
 [0.0095    ]
 [0.        ]
 [0.94696667]
 [0.9369    ]
 [0.92893333]
 [0.9326    ]
 [0.97063333]
 [0.95686667]
 [0.72766667]
 [0.01016667]
 [0.        ]
 [0.72796667]
 [0.751     ]
 [0.9429    ]
 [0.93506667]
 [0.885     ]
 [0.32096667]
 [0.0174    ]
 [0.        ]
 [0.95563333]
 [0.98136667]
 [0.97883333]
 [0.94683333]
 [0.92646667]
 [0.2737    ]
 [0.02876667]
 [0.        ]
 [0.9025    ]
 [0.9426    ]
 [0.93683333]
 [0.93753333]
 [0.9363    ]
 [0.92646667]
 [0.01706667]
 [0.        ]
 [0.90996667]
 [0.9332    ]
 [0.94056667]
 [0.93186667]
 [0.95676667]
 [0.83383333]
 [0.01043333]
 [0.  

In [26]:
y_test_scaled_df = pd.DataFrame(y_test_scaled, index=y_test.index, columns=y_test.columns)
y_test_scaled_df

,Generación
14964,0.954967
14965,0.958667
14966,0.960967
14967,0.860433
14968,0.857167
...,...
18280,0.852067
18281,0.846200
18282,0.762400
18283,0.527500


In [27]:
y_scaller_all = MinMaxScaler().fit(y)
print(y_scaller_all)

MinMaxScaler()


In [28]:
y_scaled = y_scaller_all.transform(y)
print(y_scaled)
print(y_scaled.shape)

[[0.68654263]
 [0.86996171]
 [0.95      ]
 ...
 [0.7624    ]
 [0.5275    ]
 [0.04833333]]
(3447, 1)


In [29]:
y_scaled_df = pd.DataFrame(y_scaled, index=y.index, columns=y.columns)
y_scaled_df

,Generación
37,0.686543
207,0.869962
208,0.950000
209,0.950000
210,0.950000
...,...
18280,0.852067
18281,0.846200
18282,0.762400
18283,0.527500


## Definición de modelos

### RandomForest

In [30]:
from lightgbm import LGBMRegressor
import optuna
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
import seaborn as sns
from sklearn.metrics import mean_absolute_percentage_error as mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error as mean_absolute_error
from sklearn.metrics import mean_squared_error as mean_squared_error
from sklearn.metrics import r2_score as r2_score

In [31]:
# Inicializar listas para métricas
LightGBM_model = LGBMRegressor(num_leaves=500, subsample= 0.10698460631792395, colsample_bytree= 0.7272836809565294, min_data_in_leaf= 85)
LightGBM_model.fit(X_train_scaled_df, y_train_scaled_df)
resultados = pd.DataFrame(index = y_test_scaled_df.index, columns=["LightGBM"])
#Ciclo diario de predicción
for i in range(len(X_test)):
    inicio = i * 1
    fin = inicio + 1

    X_test_seg = X_test_scaled_df.iloc[inicio:fin, :]
    y_test_seg = y_test_scaled_df.iloc[inicio:fin]

    if len(X_test_seg) < 1:
        break

    y_pred = LightGBM_model.predict(X_test_seg)
    y_pred = y_scaler.inverse_transform(y_pred.reshape(-1, 1))
    y_pred = np.clip(y_pred, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

    resultados.iloc[i, 0] = y_pred[0, 0]

[LightGBM] [Warning] min_data_in_leaf is set=85, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=85
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_data_in_leaf is set=85, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=85
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.136681 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 634
[LightGBM] [Info] Number of data points in the train set: 2412, number of used features: 8
[LightGBM] [Info] Start training from score 0.648351
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further split

In [32]:
resultados

,LightGBM
14964,27185.207433
14965,27412.001868
14966,27287.213123
14967,27286.866434
14968,25854.662495
...,...
18280,26531.749471
18281,26682.293871
18282,23121.786144
18283,14935.714871


In [33]:
predicciones = y_test.copy()
predicciones

,Generación
14964,28649.0
14965,28760.0
14966,28829.0
14967,25813.0
14968,25715.0
...,...
18280,25562.0
18281,25386.0
18282,22872.0
18283,15825.0


In [34]:
predicciones["LightGBM"] = resultados["LightGBM"]
predicciones

,Generación,LightGBM
14964,28649.0,27185.207433
14965,28760.0,27412.001868
14966,28829.0,27287.213123
14967,25813.0,27286.866434
14968,25715.0,25854.662495
...,...,...
18280,25562.0,26531.749471
18281,25386.0,26682.293871
18282,22872.0,23121.786144
18283,15825.0,14935.714871


In [35]:
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['LightGBM'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['LightGBM']):.4f}")

MAE: 1019.5988
RMSE: 1823.2087
R²: 0.9533


## Random Forest

In [36]:
from sklearn.ensemble import RandomForestRegressor

In [37]:
#Modelo LightGBM
RF_model = RandomForestRegressor(
    criterion="squared_error",
    random_state=0,
    n_estimators=400,
    min_impurity_decrease=0,
    max_depth=None,
    bootstrap=True
)
RF_model.fit(X_train_scaled_df, y_train_scaled_df)
# Inicializar listas para métricas
resultados = pd.DataFrame(index = y_test_scaled_df.index, columns=["Random Forest"])
#Ciclo diario de predicción
for i in range(len(X_test)):
    inicio = i * 1
    fin = inicio + 1

    X_test_seg = X_test_scaled_df.iloc[inicio:fin, :]
    y_test_seg = y_test_scaled_df.iloc[inicio:fin]

    if len(X_test_seg) < 1:
        break

    y_pred = RF_model.predict(X_test_seg)
    y_pred = y_scaler.inverse_transform(y_pred.reshape(-1, 1))
    y_pred = np.clip(y_pred, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

    resultados.iloc[i, 0] = y_pred[0, 0]

In [38]:
predicciones["Random Forest"] = resultados["Random Forest"]
predicciones

,Generación,LightGBM,Random Forest
14964,28649.0,27185.207433,28570.205916
14965,28760.0,27412.001868,28087.903653
14966,28829.0,27287.213123,28487.044034
14967,25813.0,27286.866434,28368.003603
14968,25715.0,25854.662495,25979.49
...,...,...,...
18280,25562.0,26531.749471,26013.520588
18281,25386.0,26682.293871,26161.99488
18282,22872.0,23121.786144,22550.322475
18283,15825.0,14935.714871,16669.129145


## Preparación redes neuronales

In [39]:
import numpy as np
import pandas as pd

def create_sliding_window_with_index(data_X, data_y, lookback):
    X, y, indices = [], [], []
    
    # Asegurar que `data_y` tiene los mismos índices que `data_X`
    data_y = data_y.reindex(data_X.index)

    max_index = len(data_X) - lookback

    for i in range(max_index):
        X.append(data_X.iloc[i:i + lookback].values)  # Ventana de entrada
        
        # Obtener el índice correcto en `data_y`
        y_index = data_X.index[i + lookback]

        # Extraer el valor correspondiente de `data_y`
        if y_index in data_y.index:
            y_value = data_y.loc[y_index]
            if isinstance(y_value, pd.Series):  # Si devuelve una serie, extraer el valor
                y_value = y_value.iloc[0]
        else:
            y_value = np.nan  # Si no está, asignamos NaN

        y.append(y_value)
        indices.append(y_index)  # 🔹 Guardamos el índice original de `data_y`

    # Convertimos `X` en un array y `y` en DataFrame conservando sus índices originales
    X_array = np.array(X)
    y_df = pd.DataFrame(y, index=indices, columns=['y'])  # 🔹 Conservamos los índices originales

    return X_array, y_df


In [40]:
lookback = 48  # Puedes ajustar a 24, 72, etc.

# Aplicar la ventana deslizante a cada conjunto
X_train_windowed, y_train_windowed = create_sliding_window_with_index(X_train_scaled_df, y_train_scaled_df, lookback)
X_val_windowed, y_val_windowed = create_sliding_window_with_index(X_val_scaled_df, y_val_scaled_df, lookback)
X_test_windowed, y_test_windowed = create_sliding_window_with_index(X_test_scaled_df, y_test_scaled_df, lookback)


In [41]:
print(f'X_train: {X_train_windowed.shape}, y_train: {y_train_windowed.shape}')
print(f'X_val: {X_val_windowed.shape}, y_val: {y_val_windowed.shape}')
print(f'X_test: {X_test_windowed.shape}, y_test: {y_test_windowed.shape}')

X_train: (2364, 48, 9), y_train: (2364, 1)
X_val: (469, 48, 9), y_val: (469, 1)
X_test: (470, 48, 9), y_test: (470, 1)


## CTNET

In [42]:
import tensorflow as tf
from tensorflow.keras import layers

In [43]:
def compile_and_fit(model, xtrain=X_train_windowed, ytrain=y_train_windowed, learning_rate=0.0001):
    model.compile(loss=[tf.keras.losses.MeanSquaredError()],
                  optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=[tf.keras.metrics.RootMeanSquaredError(), tf.keras.metrics.MeanAbsolutePercentageError(), tf.keras.metrics.MeanAbsoluteError()])
    
    history = model.fit(xtrain, ytrain, epochs=50,
                        batch_size=512, validation_split=0.2, verbose=1)
    return history

def Loss(train_loss, valid_loss):
    plt.plot(train_loss)
    plt.plot(valid_loss)
    plt.rcParams["figure.figsize"] = (15, 3)
    plt.title('Model Losses')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train Loss', 'Validation Loss'], loc='upper left')
    plt.savefig('out/loss_plot.png')
    plt.show()

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    x = layers.LayerNormalization()(inputs)
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=128, kernel_size=2, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(norm_x, norm_x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    return norm_x

def build_model(input_shape, head_size, num_heads, ff_dim, num_transformer_blocks, mlp_units, dropout=0, mlp_dropout=0):
    inputs = tf.keras.Input(shape=input_shape)
    x = inputs
    
    for _ in range(num_transformer_blocks):
        enc_out = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)
    
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(enc_out, enc_out)
    res = x + enc_out
    x = layers.LayerNormalization(epsilon=1e-6)(res)
    x = layers.GlobalAveragePooling1D(data_format="channels_first")(x)
    x = layers.Dense(832, activation="relu")(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(mlp_dropout)(x)

    outputs = layers.Dense(1)(x)
    
    return tf.keras.Model(inputs, outputs)

In [44]:
CTNET = build_model((X_train_windowed.shape[1], X_train_windowed.shape[2]), head_size=4, num_heads=3, ff_dim=32, num_transformer_blocks=3, mlp_units=[256], mlp_dropout=0.3, dropout=0.2)

In [45]:
history = compile_and_fit(CTNET)

Epoch 1/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 60s 3s/step - loss: 0.5577 - mean_absolute_error: 0.6581 - mean_absolute_percentage_error: 119361.3438 - root_mean_squared_error: 0.7468 - val_loss: 0.4494 - val_mean_absolute_error: 0.5933 - val_mean_absolute_percentage_error: 697790.1875 - val_root_mean_squared_error: 0.6704
Epoch 2/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 9s 2s/step - loss: 0.5517 - mean_absolute_error: 0.6572 - mean_absolute_percentage_error: 693157.3125 - root_mean_squared_error: 0.7427 - val_loss: 0.4396 - val_mean_absolute_error: 0.5867 - val_mean_absolute_percentage_error: 1513854.8750 - val_root_mean_squared_error: 0.6630
Epoch 3/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 15s 3s/step - loss: 0.5370 - mean_absolute_error: 0.6459 - mean_absolute_percentage_error: 1352204.8750 - root_mean_squared_error: 0.7328 - val_loss: 0.4284 - val_mean_absolute_error: 0.5792 - val_mean_absolute_percentage_error: 2459601.7500 - val_root_mean_squared_error: 0.6545
Epoch 4/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 17s 1s/step - loss: 

In [46]:
CTNET_predictions = CTNET.predict(X_test_windowed)
CTNET_predictions

15/15 ━━━━━━━━━━━━━━━━━━━━ 7s 211ms/step


array([[0.6555535 ],
       [0.657108  ],
       [0.66575843],
       [0.67059064],
       [0.67342776],
       [0.6734868 ],
       [0.6588155 ],
       [0.65137315],
       [0.6465503 ],
       [0.65660775],
       [0.68172014],
       [0.7023173 ],
       [0.7032907 ],
       [0.69911325],
       [0.6897924 ],
       [0.6595206 ],
       [0.64228725],
       [0.64479864],
       [0.66371757],
       [0.68432707],
       [0.7015709 ],
       [0.70158476],
       [0.69555736],
       [0.6653968 ],
       [0.6422406 ],
       [0.638246  ],
       [0.64928025],
       [0.6652941 ],
       [0.65014344],
       [0.6521908 ],
       [0.6711165 ],
       [0.69056696],
       [0.6881078 ],
       [0.6806459 ],
       [0.6550373 ],
       [0.6355596 ],
       [0.6199139 ],
       [0.6243264 ],
       [0.6434951 ],
       [0.668917  ],
       [0.68629766],
       [0.69210255],
       [0.67970866],
       [0.6623308 ],
       [0.6507131 ],
       [0.63970923],
       [0.65185446],
       [0.677

In [47]:
CTNET_predictions = y_scaler.inverse_transform(CTNET_predictions.reshape(-1, 1))
CTNET_predictions = np.clip(CTNET_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [48]:
resultados = pd.DataFrame(CTNET_predictions, index = y_test_windowed.index, columns=["CTNET"])

In [49]:
predicciones["CTNET"] = resultados["CTNET"]
predicciones

,Generación,LightGBM,Random Forest,CTNET
14964,28649.0,27185.207433,28570.205916,NaN
14965,28760.0,27412.001868,28087.903653,NaN
14966,28829.0,27287.213123,28487.044034,NaN
14967,25813.0,27286.866434,28368.003603,NaN
14968,25715.0,25854.662495,25979.49,NaN
...,...,...,...,...
18280,25562.0,26531.749471,26013.520588,17066.488281
18281,25386.0,26682.293871,26161.99488,17084.369141
18282,22872.0,23121.786144,22550.322475,17165.070312
18283,15825.0,14935.714871,16669.129145,17223.779297


In [50]:
predicciones["CTNET"] = predicciones["CTNET"].fillna(0)

In [51]:
# import optuna
# import tensorflow as tf
# from tensorflow.keras import layers
# from sklearn.model_selection import train_test_split

# # Definir la función objetivo para Optuna
# def objective(trial):
#     # Sugerir valores para los hiperparámetros
#     head_size = trial.suggest_int("head_size", 8, 64, step=8)
#     num_heads = trial.suggest_int("num_heads", 2, 8, step=2)
#     ff_dim = trial.suggest_int("ff_dim", 32, 256, step=32)
#     num_transformer_blocks = trial.suggest_int("num_transformer_blocks", 1, 4)
#     mlp_units = trial.suggest_categorical("mlp_units", [[128, 64], [256, 128, 64], [512, 256, 128]])
#     dropout = trial.suggest_float("dropout", 0.1, 0.5, step=0.1)
#     mlp_dropout = trial.suggest_float("mlp_dropout", 0.1, 0.5, step=0.1)
#     learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-2)

#     # Construcción del modelo con los hiperparámetros sugeridos
#     model = build_model(
#         input_shape=X_train_windowed.shape[1:],
#         head_size=head_size,
#         num_heads=num_heads,
#         ff_dim=ff_dim,
#         num_transformer_blocks=num_transformer_blocks,
#         mlp_units=mlp_units,
#         dropout=dropout,
#         mlp_dropout=mlp_dropout
#     )

#     # Compilar el modelo con los hiperparámetros sugeridos
#     model.compile(
#         loss=tf.keras.losses.MeanSquaredError(),
#         optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
#         metrics=[tf.keras.metrics.RootMeanSquaredError()]
#     )

#     # Entrenamiento con un número reducido de épocas para acelerar la búsqueda
#     history = model.fit(
#         X_train_windowed, y_train_windowed,
#         validation_split=0.2,
#         epochs=50,  # Reducimos las épocas para acelerar la búsqueda
#         batch_size=512,
#         verbose=0
#     )

#     # Obtener la métrica de validación (RMSE) y minimizarla
#     val_rmse = min(history.history["val_root_mean_squared_error"])
    
#     return val_rmse  # Queremos minimizar el RMSE

# # Ejecutar la optimización de hiperparámetros
# study = optuna.create_study(direction="minimize")
# study.optimize(objective, n_trials=20, timeout=3600)  # 20 iteraciones, máximo 1 hora

# # Mostrar los mejores hiperparámetros encontrados
# best_params = study.best_params
# print(f"Mejores hiperparámetros: {best_params}")


In [52]:
CTNET = build_model((X_train_windowed.shape[1], X_train_windowed.shape[2]), head_size=16, num_heads=8, ff_dim=256, num_transformer_blocks=1, mlp_units=[128,64], mlp_dropout=0.2, dropout=0.2)

In [53]:
history = compile_and_fit(CTNET, learning_rate = 0.0010762230908145116)

Epoch 1/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 72s 4s/step - loss: 0.5277 - mean_absolute_error: 0.6353 - mean_absolute_percentage_error: 1164696.1250 - root_mean_squared_error: 0.7264 - val_loss: 0.3474 - val_mean_absolute_error: 0.5249 - val_mean_absolute_percentage_error: 9804954.0000 - val_root_mean_squared_error: 0.5894
Epoch 2/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 18s 3s/step - loss: 0.3916 - mean_absolute_error: 0.5519 - mean_absolute_percentage_error: 11790484.0000 - root_mean_squared_error: 0.6253 - val_loss: 0.1535 - val_mean_absolute_error: 0.3634 - val_mean_absolute_percentage_error: 35614028.0000 - val_root_mean_squared_error: 0.3918
Epoch 3/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 25s 4s/step - loss: 0.1754 - mean_absolute_error: 0.3769 - mean_absolute_percentage_error: 36111816.0000 - root_mean_squared_error: 0.4185 - val_loss: 0.1772 - val_mean_absolute_error: 0.2947 - val_mean_absolute_percentage_error: 85370072.0000 - val_root_mean_squared_error: 0.4209
Epoch 4/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 22s 4s/step 

In [54]:
CTNET_predictions = CTNET.predict(X_test_windowed)
CTNET_predictions

15/15 ━━━━━━━━━━━━━━━━━━━━ 23s 757ms/step


array([[0.65497905],
       [0.88238037],
       [0.8850447 ],
       [0.8785308 ],
       [0.8704098 ],
       [0.82187617],
       [0.18902595],
       [0.1789166 ],
       [0.53652114],
       [0.87205106],
       [0.9095257 ],
       [0.9113905 ],
       [0.8983334 ],
       [0.8900463 ],
       [0.85778636],
       [0.17602475],
       [0.35612938],
       [0.87855446],
       [0.8983943 ],
       [0.8976761 ],
       [0.8910093 ],
       [0.8879427 ],
       [0.87108535],
       [0.22102083],
       [0.64406234],
       [0.89495546],
       [0.89144117],
       [0.878815  ],
       [0.21772534],
       [0.7048357 ],
       [0.92202634],
       [0.9162462 ],
       [0.8916416 ],
       [0.85927767],
       [0.21677236],
       [0.21813507],
       [0.51766574],
       [0.8932926 ],
       [0.9005584 ],
       [0.9029886 ],
       [0.8976671 ],
       [0.8823766 ],
       [0.8185838 ],
       [0.37822372],
       [0.20419495],
       [0.20817235],
       [0.90178555],
       [0.927

In [55]:
CTNET_predictions = y_scaler.inverse_transform(CTNET_predictions.reshape(-1, 1))
CTNET_predictions = np.clip(CTNET_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [56]:
resultados = pd.DataFrame(CTNET_predictions, index = y_test_windowed.index, columns=["CTNET"])

In [57]:
predicciones["CTNET"] = resultados["CTNET"]
predicciones

,Generación,LightGBM,Random Forest,CTNET
14964,28649.0,27185.207433,28570.205916,NaN
14965,28760.0,27412.001868,28087.903653,NaN
14966,28829.0,27287.213123,28487.044034,NaN
14967,25813.0,27286.866434,28368.003603,NaN
14968,25715.0,25854.662495,25979.49,NaN
...,...,...,...,...
18280,25562.0,26531.749471,26013.520588,25993.050781
18281,25386.0,26682.293871,26161.99488,24743.876953
18282,22872.0,23121.786144,22550.322475,23200.759766
18283,15825.0,14935.714871,16669.129145,14345.122070


## Forecasting

In [58]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.metrics import RootMeanSquaredError
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.losses import Huber
from tensorflow.keras.callbacks import EarlyStopping

In [59]:
Forecast_model = Sequential()
Forecast_model.add(InputLayer((X_train_windowed.shape[1], X_train_windowed.shape[2])))

#CNN
Forecast_model.add(Conv1D(filters=64, kernel_size=2, padding='same', activation='relu'))
Forecast_model.add(BatchNormalization())  # 🔹 Nueva Normalización aquí
Forecast_model.add(MaxPooling1D(pool_size=2))

#model_Soleado.add(Flatten())
#BiLSTM
Forecast_model.add(Bidirectional(LSTM(128, return_sequences=True)))
Forecast_model.add(Bidirectional(LSTM(64, return_sequences=True)))
Forecast_model.add(Dropout(0.2))  # 🔹 Mayor regularización en BiLSTM
Forecast_model.add(Bidirectional(LSTM(32, return_sequences=False)))

#Normalización y Dropout
Forecast_model.add(BatchNormalization())
Forecast_model.add(Dropout(0.3))

# Capas Densas
Forecast_model.add(Dense(16, activation='relu'))
Forecast_model.add(Dense(1, 'relu'))

Forecast_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_12 (Conv1D)              │ (None, 48, 64)         │         1,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 48, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 24, 256)        │       197,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 24, 128)        │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 24, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 16)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 405,985 (1.55 MB)

 Trainable params: 405,729 (1.55 MB)

 Non-trainable params: 256 (1.00 KB)

In [60]:
cp = ModelCheckpoint('Forcasting_model.keras', save_best_only=True)
Forecast_model.compile(optimizer=Adam(learning_rate=0.0001), loss=Huber(delta=1000), metrics=['mae'])
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [61]:
history = Forecast_model.fit(X_train_windowed, y_train_windowed, validation_data=(X_val_windowed, y_val_windowed), epochs=100, batch_size=8, callbacks=[cp, early_stop])

Epoch 1/100


296/296 ━━━━━━━━━━━━━━━━━━━━ 335s 368ms/step - loss: 0.2016 - mae: 0.5333 - val_loss: 0.2066 - val_mae: 0.5816
Epoch 2/100
296/296 ━━━━━━━━━━━━━━━━━━━━ 115s 367ms/step - loss: 0.1407 - mae: 0.4360 - val_loss: 0.0898 - val_mae: 0.3470
Epoch 3/100
296/296 ━━━━━━━━━━━━━━━━━━━━ 143s 362ms/step - loss: 0.1054 - mae: 0.3690 - val_loss: 0.0950 - val_mae: 0.3514
Epoch 4/100
296/296 ━━━━━━━━━━━━━━━━━━━━ 132s 316ms/step - loss: 0.0955 - mae: 0.3496 - val_loss: 0.0624 - val_mae: 0.2526
Epoch 5/100
296/296 ━━━━━━━━━━━━━━━━━━━━ 144s 311ms/step - loss: 0.0858 - mae: 0.3325 - val_loss: 0.0542 - val_mae: 0.2401
Epoch 6/100
296/296 ━━━━━━━━━━━━━━━━━━━━ 144s 306ms/step - loss: 0.0831 - mae: 0.3236 - val_loss: 0.0463 - val_mae: 0.2290
Epoch 7/100
296/296 ━━━━━━━━━━━━━━━━━━━━ 184s 433ms/step - loss: 0.0795 - mae: 0.3147 - val_loss: 0.0418 - val_mae: 0.2132
Epoch 8/100
296/296 ━━━━━━━━━━━━━━━━━━━━ 132s 385ms/step - loss: 0.0709 - mae: 0.2955 - val_loss: 0.0416 - val_mae: 0.2209
Epoch 9/100
296/296 ━━━━━━━━

In [62]:
Forecast_predictions = Forecast_model.predict(X_test_windowed)
Forecast_predictions

14/15 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step

15/15 ━━━━━━━━━━━━━━━━━━━━ 79s 3s/step 


array([[0.78187525],
       [0.85450816],
       [0.84117824],
       [0.8430827 ],
       [0.8088846 ],
       [0.78877974],
       [0.17309554],
       [0.15284581],
       [0.72884065],
       [0.85092837],
       [0.8398391 ],
       [0.8487372 ],
       [0.82202095],
       [0.8167394 ],
       [0.7797253 ],
       [0.        ],
       [0.7344668 ],
       [0.82965595],
       [0.83172745],
       [0.82951665],
       [0.8117194 ],
       [0.79736173],
       [0.7722986 ],
       [0.        ],
       [0.82628244],
       [0.82531345],
       [0.822931  ],
       [0.80492985],
       [0.35324183],
       [0.8495433 ],
       [0.84404427],
       [0.8214418 ],
       [0.81820446],
       [0.7749531 ],
       [0.32811058],
       [0.30248684],
       [0.74913025],
       [0.7777006 ],
       [0.77432567],
       [0.80689245],
       [0.7765493 ],
       [0.7603438 ],
       [0.6396581 ],
       [0.42623   ],
       [0.16389963],
       [0.01606916],
       [0.79341495],
       [0.822

In [63]:
Forecast_predictions = y_scaler.inverse_transform(Forecast_predictions.reshape(-1, 1))
Forecast_predictions = np.clip(Forecast_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [64]:
Forecast_resultados = pd.DataFrame(Forecast_predictions, index = y_test_windowed.index, columns=["Forecast"])

In [65]:
predicciones["Forecast"] = Forecast_resultados["Forecast"]
predicciones

,Generación,LightGBM,Random Forest,CTNET,Forecast
14964,28649.0,27185.207433,28570.205916,NaN,NaN
14965,28760.0,27412.001868,28087.903653,NaN,NaN
14966,28829.0,27287.213123,28487.044034,NaN,NaN
14967,25813.0,27286.866434,28368.003603,NaN,NaN
14968,25715.0,25854.662495,25979.49,NaN,NaN
...,...,...,...,...,...
18280,25562.0,26531.749471,26013.520588,25993.050781,24631.632812
18281,25386.0,26682.293871,26161.99488,24743.876953,23384.947266
18282,22872.0,23121.786144,22550.322475,23200.759766,22358.246094
18283,15825.0,14935.714871,16669.129145,14345.122070,17097.496094


## Métricas

In [66]:
predicciones.loc[~predicciones['CTNET'].isna(),'Generación']

15084    28669.0
15085    29441.0
15086    29365.0
15087    28405.0
15088    27794.0
          ...   
18280    25562.0
18281    25386.0
18282    22872.0
18283    15825.0
18284     1450.0
Name: Generación, Length: 470, dtype: float64

## Photovoltaic

In [67]:
from tensorflow.keras.models import Model
inputs = Input(shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]))

# Primera capa CNN
x = Conv1D(filters=64, kernel_size=4, padding='same', activation='relu')(inputs)
x = MaxPooling1D(pool_size=2)(x)

# Segunda capa CNN
x = Conv1D(filters=128, kernel_size=4, padding='same', activation='relu')(x)
x = MaxPooling1D(pool_size=2)(x)

# Capa BiGRU
x = Bidirectional(GRU(64, return_sequences=True))(x)

# Atención: se define de forma explícita
attention = MultiHeadAttention(num_heads=4, key_dim=128)(x, x)

# Aplanar y agregar Dropout
x = Flatten()(attention)
x = Dropout(0.4)(x)
initializer = tf.keras.initializers.HeNormal()
x = Dense(64, activation="relu", kernel_regularizer=l2(0.01))(x)
x = Dense(32, activation="relu")(x)  # Otra capa intermedia

# Capa de salida
outputs = Dense(1, activation="linear")(x)

# Definir el modelo
Photo_model = Model(inputs=inputs, outputs=outputs)

# Resumen del modelo
Photo_model.summary()

Model: "functional_13"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 48, 9)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_13 (Conv1D)  │ (None, 48, 64)    │      2,368 │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 24, 64)    │          0 │ conv1d_13[0][0]   │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_14 (Conv1D)  │ (None, 24, 128)   │     32,896 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_2     │ (None, 12, 128)   │          0 │ conv1d_14[0][0]   │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_3     │ (None, 12, 128)   │     74,496 │ max_pooling1d_2[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 12, 128)   │    263,808 │ bidirectional_3[… │
│ (MultiHeadAttentio… │                   │            │ bidirectional_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 1536)      │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_11          │ (None, 1536)      │          0 │ flatten[0][0]     │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 64)        │     98,368 │ dropout_11[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 32)        │      2,080 │ dense_10[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 1)         │         33 │ dense_11[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 474,049 (1.81 MB)

 Trainable params: 474,049 (1.81 MB)

 Non-trainable params: 0 (0.00 B)

In [68]:
cp2 = ModelCheckpoint('Photovoltaic_model.keras', save_best_only=True)
Photo_model.compile(optimizer=Adam(learning_rate=0.0001), loss="mean_squared_error", metrics=['mae'])
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [69]:
history = Photo_model.fit(
    X_train_windowed, y_train_windowed,
    validation_data=(X_val_windowed, y_val_windowed),
    epochs=50,
    batch_size=16,
    callbacks=[cp, early_stop]
)

Epoch 1/50
148/148 ━━━━━━━━━━━━━━━━━━━━ 290s 469ms/step - loss: 1.3068 - mae: 0.3753 - val_loss: 0.8472 - val_mae: 0.2702
Epoch 2/50
148/148 ━━━━━━━━━━━━━━━━━━━━ 59s 288ms/step - loss: 0.7673 - mae: 0.2646 - val_loss: 0.5581 - val_mae: 0.2503
Epoch 3/50
148/148 ━━━━━━━━━━━━━━━━━━━━ 78s 218ms/step - loss: 0.5081 - mae: 0.2566 - val_loss: 0.3811 - val_mae: 0.2246
Epoch 4/50
148/148 ━━━━━━━━━━━━━━━━━━━━ 53s 275ms/step - loss: 0.3420 - mae: 0.2336 - val_loss: 0.2541 - val_mae: 0.2060
Epoch 5/50
148/148 ━━━━━━━━━━━━━━━━━━━━ 83s 242ms/step - loss: 0.2289 - mae: 0.1952 - val_loss: 0.1967 - val_mae: 0.2041
Epoch 6/50
148/148 ━━━━━━━━━━━━━━━━━━━━ 41s 241ms/step - loss: 0.1633 - mae: 0.1742 - val_loss: 0.1433 - val_mae: 0.1877
Epoch 7/50
148/148 ━━━━━━━━━━━━━━━━━━━━ 43s 247ms/step - loss: 0.1249 - mae: 0.1653 - val_loss: 0.1219 - val_mae: 0.1800
Epoch 8/50
148/148 ━━━━━━━━━━━━━━━━━━━━ 55s 307ms/step - loss: 0.1017 - mae: 0.1619 - val_loss: 0.1033 - val_mae: 0.1849
Epoch 9/50
148/148 ━━━━━━━━━━━━

In [70]:
Photo_predictions = Photo_model.predict(X_test_windowed)
Photo_predictions

15/15 ━━━━━━━━━━━━━━━━━━━━ 42s 1s/step


array([[0.8017259 ],
       [0.9155079 ],
       [0.88351774],
       [0.898451  ],
       [0.8344611 ],
       [0.8089581 ],
       [0.29764456],
       [0.3978417 ],
       [0.7545711 ],
       [0.90653104],
       [0.9168828 ],
       [0.95017827],
       [0.8749721 ],
       [0.8391649 ],
       [0.8362489 ],
       [0.16638964],
       [0.80093336],
       [0.9213102 ],
       [0.95709425],
       [0.9503931 ],
       [0.8900832 ],
       [0.8243661 ],
       [0.84515935],
       [0.28722587],
       [0.9013023 ],
       [0.9299159 ],
       [0.96670616],
       [0.9469912 ],
       [0.48739526],
       [0.94166225],
       [1.0354537 ],
       [0.9799162 ],
       [0.941506  ],
       [0.8552394 ],
       [0.35310417],
       [0.4768762 ],
       [0.7904162 ],
       [0.9710121 ],
       [1.0097541 ],
       [1.0197252 ],
       [0.8331157 ],
       [0.810413  ],
       [0.6927415 ],
       [0.48355907],
       [0.42149493],
       [0.21635062],
       [0.8669723 ],
       [0.919

In [71]:
Photo_predictions = y_scaler.inverse_transform(Photo_predictions.reshape(-1, 1))
Photo_predictions = np.clip(Photo_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [72]:
Photo_resultados = pd.DataFrame(Photo_predictions, index = y_test_windowed.index, columns=["Photo"])

In [73]:
predicciones["Photo"] = Photo_resultados["Photo"]
predicciones

,Generación,LightGBM,Random Forest,CTNET,Forecast,Photo
14964,28649.0,27185.207433,28570.205916,NaN,NaN,NaN
14965,28760.0,27412.001868,28087.903653,NaN,NaN,NaN
14966,28829.0,27287.213123,28487.044034,NaN,NaN,NaN
14967,25813.0,27286.866434,28368.003603,NaN,NaN,NaN
14968,25715.0,25854.662495,25979.49,NaN,NaN,NaN
...,...,...,...,...,...,...
18280,25562.0,26531.749471,26013.520588,25993.050781,24631.632812,23882.898438
18281,25386.0,26682.293871,26161.99488,24743.876953,23384.947266,21068.599609
18282,22872.0,23121.786144,22550.322475,23200.759766,22358.246094,18626.800781
18283,15825.0,14935.714871,16669.129145,14345.122070,17097.496094,16857.214844


In [74]:
print("LightGBM")
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['LightGBM'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print("Random Forest")
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['Random Forest']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['Random Forest'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['Random Forest']):.4f}")
print("CTNET")
print(f"MAE: {mean_absolute_error(predicciones.loc[~predicciones['CTNET'].isna(),'Generación'], predicciones.loc[~predicciones['CTNET'].isna(),'CTNET']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones.loc[~predicciones['CTNET'].isna(),'Generación'], predicciones.loc[~predicciones['CTNET'].isna(),'CTNET'])):.4f}")
print(f"R²: {r2_score(predicciones.loc[~predicciones['CTNET'].isna(),'Generación'], predicciones.loc[~predicciones['CTNET'].isna(),'CTNET']):.4f}")
print("Forecast")
print(f"MAE: {mean_absolute_error(predicciones.loc[~predicciones['Forecast'].isna(),'Generación'], predicciones.loc[~predicciones['Forecast'].isna(),'Forecast']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones.loc[~predicciones['Forecast'].isna(),'Generación'], predicciones.loc[~predicciones['Forecast'].isna(),'Forecast'])):.4f}")
print(f"R²: {r2_score(predicciones.loc[~predicciones['Forecast'].isna(),'Generación'], predicciones.loc[~predicciones['Forecast'].isna(),'Forecast']):.4f}")
print("Photovoltaic")
print(f"MAE: {mean_absolute_error(predicciones.loc[~predicciones['Photo'].isna(),'Generación'], predicciones.loc[~predicciones['Photo'].isna(),'Photo']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones.loc[~predicciones['Photo'].isna(),'Generación'], predicciones.loc[~predicciones['Photo'].isna(),'Photo'])):.4f}")
print(f"R²: {r2_score(predicciones.loc[~predicciones['Photo'].isna(),'Generación'], predicciones.loc[~predicciones['Photo'].isna(),'Photo']):.4f}")

LightGBM
MAE: 1019.5988
RMSE: 1823.2087
R²: 0.9533
Random Forest
MAE: 1132.4972
RMSE: 2125.0614
R²: 0.9366
CTNET
MAE: 3903.6518
RMSE: 6640.9185
R²: 0.3338
Forecast
MAE: 4584.1002
RMSE: 6644.8977
R²: 0.3330
Photovoltaic
MAE: 4021.5300
RMSE: 6103.0076
R²: 0.4374


In [75]:
# Seleccionar las columnas desde "LightGBM" en adelante
columnas_nuevas = predicciones.loc[:, "LightGBM":]

# Unir con `datos` usando el índice, manteniendo todo en `datos`
datos = datos.merge(columnas_nuevas, left_index=True, right_index=True, how='left')

# Ver resultado
datos.head()


,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day,LightGBM,Random Forest,CTNET,Forecast,Photo
24,2022-09-02 00:00:00,0.0,19,6,76,0,4,15,0,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
25,2022-09-02 01:00:00,0.0,18,7,81,0,4,15,1,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
26,2022-09-02 02:00:00,0.0,18,7,84,0,4,15,2,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
27,2022-09-02 03:00:00,0.0,18,7,86,0,4,15,3,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
28,2022-09-02 04:00:00,0.0,17,7,86,0,4,15,4,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN


## X_train para hacer análisis de sobreajuste

In [76]:
predicciones_train = y_train.copy()

In [77]:
LightGBM_predictions_train = LightGBM_model.predict(X_train_scaled_df)
LightGBM_predictions_train = y_scaler.inverse_transform(LightGBM_predictions_train.reshape(-1, 1))
LightGBM_predictions_train = np.clip(LightGBM_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
LightGBM_resultados = pd.DataFrame(LightGBM_predictions_train, index = y_train_scaled_df.index, columns=["LightGBM_train"])
predicciones_train["LightGBM_train"] = LightGBM_resultados["LightGBM_train"]

[LightGBM] [Warning] min_data_in_leaf is set=85, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=85


In [78]:
RandomForest_predictions_train = RF_model.predict(X_train_scaled_df)
RandomForest_predictions_train = y_scaler.inverse_transform(RandomForest_predictions_train.reshape(-1, 1))
RandomForest_predictions_train = np.clip(RandomForest_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
RandomForest_resultados = pd.DataFrame(RandomForest_predictions_train, index = y_train_scaled_df.index, columns=["RandomForest_train"])
predicciones_train["RandomForest_train"] = RandomForest_resultados["RandomForest_train"]

In [79]:
CTNET_predictions_train = CTNET.predict(X_train_windowed)
CTNET_predictions_train = y_scaler.inverse_transform(CTNET_predictions_train.reshape(-1, 1))
CTNET_predictions_train = np.clip(CTNET_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
CTNET_resultados = pd.DataFrame(CTNET_predictions_train, index = y_train_windowed.index, columns=["CTNET_train"])
predicciones_train["CTNET_train"] = CTNET_resultados["CTNET_train"]

74/74 ━━━━━━━━━━━━━━━━━━━━ 18s 121ms/step


In [80]:
Forecast_predictions_train = Forecast_model.predict(X_train_windowed)
Forecast_predictions_train = y_scaler.inverse_transform(Forecast_predictions_train.reshape(-1, 1))
Forecast_predictions_train = np.clip(Forecast_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
Forecast_resultados = pd.DataFrame(Forecast_predictions_train, index = y_train_windowed.index, columns=["Forecast_train"])
predicciones_train["Forecast_train"] = Forecast_resultados["Forecast_train"]

74/74 ━━━━━━━━━━━━━━━━━━━━ 22s 172ms/step


In [81]:
Photo_predictions_train = Photo_model.predict(X_train_windowed)
Photo_predictions_train = y_scaler.inverse_transform(Photo_predictions_train.reshape(-1, 1))
Photo_predictions_train = np.clip(Photo_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
Photo_resultados = pd.DataFrame(Photo_predictions_train, index = y_train_windowed.index, columns=["Photo_train"])
predicciones_train["Photo_train"] = Photo_resultados["Photo_train"]

74/74 ━━━━━━━━━━━━━━━━━━━━ 12s 64ms/step


In [82]:
predicciones_train

,Generación,LightGBM_train,RandomForest_train,CTNET_train,Forecast_train,Photo_train
37,20596.278869,25889.087712,23128.419174,NaN,NaN,NaN
207,26098.851182,24843.499368,25386.898087,NaN,NaN,NaN
208,28500.000000,26328.342640,27553.583407,NaN,NaN,NaN
209,28500.000000,24809.626208,28066.727050,NaN,NaN,NaN
210,28500.000000,25409.045786,27100.040132,NaN,NaN,NaN
...,...,...,...,...,...,...
12286,0.000000,0.000000,0.000000,17745.292969,2115.510498,9332.243164
12301,22808.000000,23232.824009,23146.920008,22325.748047,17512.943359,17208.289062
12302,21959.000000,21738.836363,21760.326689,24774.064453,22042.119141,19113.902344
12303,21047.000000,20951.472475,20928.136453,24125.187500,20931.369141,21776.238281


In [83]:
print("LightGBM")
print(f"MAE: {mean_absolute_error(predicciones_train['Generación'], predicciones_train['LightGBM_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train['Generación'], predicciones_train['LightGBM_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train['Generación'], predicciones_train['LightGBM_train']):.4f}")
print("Random Forest")
print(f"MAE: {mean_absolute_error(predicciones_train['Generación'], predicciones_train['RandomForest_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train['Generación'], predicciones_train['RandomForest_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train['Generación'], predicciones_train['RandomForest_train']):.4f}")
print("CTNET")
print(f"MAE: {mean_absolute_error(predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'CTNET_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'CTNET_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'CTNET_train']):.4f}")
print("Forecast")
print(f"MAE: {mean_absolute_error(predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Forecast_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Forecast_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Forecast_train']):.4f}")
print("Photovoltaic")
print(f"MAE: {mean_absolute_error(predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Photo_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Photo_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Photo_train']):.4f}")

LightGBM
MAE: 883.6183
RMSE: 1507.0696
R²: 0.9789
Random Forest
MAE: 339.1511
RMSE: 632.8372
R²: 0.9963
CTNET
MAE: 3800.2465
RMSE: 5931.6700
R²: 0.6752
Forecast
MAE: 4660.4532
RMSE: 6905.4709
R²: 0.5599
Photovoltaic
MAE: 3791.8976
RMSE: 5738.7618
R²: 0.6960


In [84]:
# Seleccionar las columnas desde "LightGBM" en adelante
columnas_nuevas = predicciones_train.loc[:, "LightGBM_train":]

# Unir con `datos` usando el índice, manteniendo todo en `datos`
datos = datos.merge(columnas_nuevas, left_index=True, right_index=True, how='left')

# Ver resultado
datos.head()


,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,...,LightGBM,Random Forest,CTNET,Forecast,Photo,LightGBM_train,RandomForest_train,CTNET_train,Forecast_train,Photo_train
24,2022-09-02 00:00:00,0.0,19,6,76,0,4,15,0,Noche,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25,2022-09-02 01:00:00,0.0,18,7,81,0,4,15,1,Noche,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26,2022-09-02 02:00:00,0.0,18,7,84,0,4,15,2,Noche,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2022-09-02 03:00:00,0.0,18,7,86,0,4,15,3,Noche,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28,2022-09-02 04:00:00,0.0,17,7,86,0,4,15,4,Noche,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [85]:
datos.to_excel("04.4_Predicciones_Conjunto_soleado GMM.xlsx", index=True)

## Guardamos los modelos

In [86]:
import joblib

# Guardar modelo LightGBM
joblib.dump(LightGBM_model, "4_4_LightGBM_model.pkl")

# Guardar modelo Random Forest
joblib.dump(RF_model, "4_4_RandomForest_model.pkl")


['4_4_RandomForest_model.pkl']

In [87]:
CTNET.save("4_4_CTNET_model.keras")
Forecast_model.save("4_4_Forecast_model.keras")
Photo_model.save("4_4_Photo_model.keras")